# Phage Isolates: Contig Catalogue

All saved SPAdes contigs (>=1 kb) remain in the catalogue. Retained/excluded is a coverage-and-assigned-host decision, not a geNomad-positive filter. Sample selection runs independently of global clustering/annotations. Catalogue generation and final reporting collect those results without invalidating unrelated samples' mapping.

In [ ]:
from __future__ import annotations

import html
import json
import gzip
import math
import os
import re
import shutil
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


STATUS_ORDER = {"PASS": 0, "OK": 0, "INFO": 0, "WARN": 1, "REVIEW": 1, "FAIL": 2}
CHECKV_GOOD = {"Complete", "High-quality", "Medium-quality"}
CHECKV_COMPLETE = {"Complete", "High-quality"}
CONTIG_SAMPLES = {}
BLAST_FIELDS = ["query", "subject", "subject_title", "q_start", "q_end", "q_len", "s_len",
                "qcovs", "evalue", "align_len", "pident", "s_start", "s_end", "bitscore", "btop"]
from IPython.display import Image, Markdown, display


In [ ]:
def _setup_plots() -> None:
    sns.set_style("ticks", {"axes.grid": True})
    sns.set_palette("colorblind")
    plt.rcParams["axes.linewidth"] = 1.5
    plt.rcParams["xtick.major.width"] = 1.5
    plt.rcParams["ytick.major.width"] = 1.5
    plt.rcParams["xtick.major.size"] = 8
    plt.rcParams["ytick.major.size"] = 8
    plt.rcParams["axes.titlepad"] = 16
    plt.rcParams["axes.titlesize"] = 18
    plt.rcParams["axes.labelsize"] = 14
    plt.rcParams["xtick.labelsize"] = 10
    plt.rcParams["ytick.labelsize"] = 10
    plt.rcParams["legend.fontsize"] = 10
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Liberation Sans", "DejaVu Sans"]
    plt.rcParams["text.usetex"] = False
    plt.rcParams["svg.fonttype"] = "none"
    plt.rcParams["savefig.dpi"] = 300


def _as_list(value) -> list:
    if value is None:
        return []
    if isinstance(value, (str, os.PathLike)):
        return [str(value)]
    try:
        return [str(x) for x in value]
    except TypeError:
        return [str(value)]


def _ensure_parent(*paths) -> None:
    for path in paths:
        if path:
            Path(str(path)).parent.mkdir(parents=True, exist_ok=True)


def _read_table(path, sep="\t", names=None, header="infer", dtype=None) -> pd.DataFrame:
    path = Path(str(path))
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame(columns=names or [])
    try:
        return pd.read_csv(path, sep=sep, names=names, header=header, dtype=dtype)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=names or [])


def _to_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def _first_present(df: pd.DataFrame, candidates: list[str]):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None


def _clean_sample_name(value: str, sampling: str = "tot") -> str:
    sample = os.path.basename(str(value))
    replacements = [
        ".fastq.gz",
        ".fq.gz",
        ".fastq",
        ".fq",
        ".fasta",
        ".fa",
        ".fna",
        ".csv",
        ".tsv",
        ".txt",
        f"_spades_filtered_scaffolds.{sampling}",
        f"_spades_scaffolds.{sampling}",
        f"_assembled_contigs.{sampling}",
        f"_viral_contigs.{sampling}",
        f"_unfiltered_contigs.{sampling}",
        f"_{sampling}",
    ]
    for suffix in replacements:
        if sample.endswith(suffix):
            sample = sample[: -len(suffix)]
    return sample


def sample_from_contig(contig: str) -> str:
    contig = str(contig).split("|")[0]
    if contig in CONTIG_SAMPLES:
        return CONTIG_SAMPLES[contig]
    parent = re.split(r"(?:_|-)provirus_", contig, maxsplit=1)[0]
    if parent in CONTIG_SAMPLES:
        return CONTIG_SAMPLES[parent]
    named = re.fullmatch(r"(.+)_[^_]+_\d+_len_\d+", parent)
    if named:
        return named.group(1)
    if "_NODE_" in contig:
        return contig.split("_NODE_")[0]
    if "_provirus_" in contig:
        return contig.split("_provirus_")[0]
    if "-provirus_" in contig:
        return contig.split("-provirus_")[0]
    if "_contig_" in contig:
        return contig.split("_contig_")[0]
    return contig.split("|")[0]


def _concat_nonempty(frames: list[pd.DataFrame], columns: list[str] | None = None) -> pd.DataFrame:
    frames = [frame for frame in frames if frame is not None and not frame.empty]
    if not frames:
        return pd.DataFrame(columns=columns or [])
    return pd.concat(frames, ignore_index=True)


def _load_sample_host_map(results_dir: str | None, paths=None) -> dict[str, str]:
    candidates = paths if paths is not None else [
        Path(str(results_dir)) / "host_mapping_file.tsv",
        Path(str(results_dir)) / "HOST" / "host_mapping_file.tsv",
    ] if results_dir else []
    assignments = {}
    for path in candidates:
        frame = _read_table(path, dtype=str)
        lower = {str(col).strip().lower(): col for col in frame.columns}
        if "sample" in lower and "host" in lower:
            assignments.update(zip(frame[lower["sample"]].astype(str).str.strip(),
                                   frame[lower["host"]].astype(str).str.strip()))
    return assignments


In [ ]:
def load_checkv(path: str) -> pd.DataFrame:
    names = [
        "contig",
        "contig_length",
        "provirus",
        "proviral_length",
        "gene_count",
        "viral_genes",
        "host_genes",
        "checkv_quality",
        "miuvig_quality",
        "completeness",
        "completeness_method",
        "contamination",
        "kmer_freq",
        "warnings",
    ]
    df = _read_table(path, sep="\t", names=names, header=None)
    if df.empty:
        return df
    if str(df.iloc[0]["contig"]).lower() in {"contig_id", "contig"}:
        df = df.iloc[1:].copy()
    df["sample"] = df["contig"].apply(sample_from_contig)
    return _to_numeric(
        df,
        [
            "contig_length",
            "proviral_length",
            "gene_count",
            "viral_genes",
            "host_genes",
            "completeness",
            "contamination",
            "kmer_freq",
        ],
    )


def _parse_covstats_name(path: Path, reference_set: str, sampling: str) -> tuple[str, str | None]:
    name = path.name
    host_match = re.match(r"bowtie2(?:_filtered)?_(.+?)_vs_(.+?)(?:_masked_prophages)?_covstats\.txt$", name)
    if host_match and "/HOST/" in str(path):
        return host_match.group(1), host_match.group(2)

    patterns = [
        rf"bowtie2_(.+?)_assembled_contigs\.{re.escape(sampling)}_covstats\.txt$",
        rf"bowtie2_(.+?)_viral_contigs\.{re.escape(sampling)}_covstats\.txt$",
        rf"bowtie2_(.+?)_unfiltered_contigs\.{re.escape(sampling)}_covstats\.txt$",
        rf"bowtie2_(.+?)_{re.escape(sampling)}_covstats\.txt$",
    ]
    for pattern in patterns:
        match = re.match(pattern, name)
        if match:
            return match.group(1), None
    return _clean_sample_name(name, sampling=sampling), None


def parse_covstats_file(path: str, reference_set: str, sampling: str) -> pd.DataFrame:
    path = Path(str(path))
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    df = _read_table(path, sep="\t")
    if df.empty:
        return pd.DataFrame()

    contig_col = _first_present(df, ["Contig", "contig", "Sequence"])
    if not contig_col:
        return pd.DataFrame()

    mean_col = _first_present(df, [col for col in df.columns if str(col).endswith(" Mean") and not str(col).endswith(" Trimmed Mean")] + ["Mean", "mean"])
    length_col = _first_present(df, [col for col in df.columns if str(col).endswith(" Length")] + ["Length", "length"])
    covered_col = _first_present(df, [col for col in df.columns if str(col).endswith(" Covered Bases")] + ["Covered Bases", "covered_bases"])
    read_count_col = _first_present(df, [col for col in df.columns if str(col).endswith(" Read Count")] + ["Read Count"])
    trimmed_mean_col = _first_present(df, [col for col in df.columns if str(col).endswith(" Trimmed Mean")] + ["Trimmed Mean"])
    rpkm_col = _first_present(df, [col for col in df.columns if str(col).endswith(" RPKM")] + ["RPKM"])

    out = pd.DataFrame({"contig": df[contig_col].astype(str)})
    out["sample"], host = _parse_covstats_name(path, reference_set=reference_set, sampling=sampling)
    out["host"] = host
    out["reference_set"] = reference_set
    out["mean_coverage"] = pd.to_numeric(df[mean_col], errors="coerce") if mean_col else np.nan
    out["length"] = pd.to_numeric(df[length_col], errors="coerce") if length_col else np.nan
    out["covered_bases"] = pd.to_numeric(df[covered_col], errors="coerce") if covered_col else np.nan
    out["read_count"] = pd.to_numeric(df[read_count_col], errors="coerce") if read_count_col else np.nan
    out["trimmed_mean"] = pd.to_numeric(df[trimmed_mean_col], errors="coerce") if trimmed_mean_col else np.nan
    out["rpkm"] = pd.to_numeric(df[rpkm_col], errors="coerce") if rpkm_col else np.nan
    out["breadth_percent"] = out["covered_bases"] * 100.0 / out["length"].replace(0, np.nan)
    return out


def load_covstats(paths, reference_set: str, sampling: str) -> pd.DataFrame:
    frames = [parse_covstats_file(path, reference_set=reference_set, sampling=sampling) for path in _as_list(paths)]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        return pd.DataFrame(
            columns=[
                "contig",
                "sample",
                "host",
                "reference_set",
                "mean_coverage",
                "length",
                "covered_bases",
                "read_count",
                "trimmed_mean",
                "rpkm",
                "breadth_percent",
            ]
        )
    return pd.concat(frames, ignore_index=True)


def _merge_intervals(intervals):
    # Inclusive coordinates: an overlap contributes covered bases only once.
    merged = []
    for start, end in sorted((min(a, b), max(a, b)) for a, b in intervals):
        if merged and start <= merged[-1][1] + 1:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((start, end))
    return merged


def _uncovered_intervals(start, end, covered):
    remaining = []
    for left, right in covered:
        if right < start:
            continue
        if left > end:
            break
        if left > start:
            remaining.append((start, left - 1))
        start = max(start, right + 1)
    if start <= end:
        remaining.append((start, end))
    return remaining


def _blast_alignment_summary(alignments):
    # Resolve query overlaps by BLAST score; identity uses exactly those bases.
    alignments = alignments.sort_values(
        ["bitscore", "evalue", "align_len", "pident", "q_start", "q_end", "subject"],
        ascending=[False, True, False, False, True, True, True], na_position="last")
    covered = []
    matches = alignment_columns = 0.0
    traceback = True
    for row in alignments.itertuples():
        start, end = sorted([int(row.q_start), int(row.q_end)])
        remaining = _uncovered_intervals(start, end, covered)
        bases = sum(right - left + 1 for left, right in remaining)
        if not bases:
            continue
        if pd.notna(row.btop):
            position = int(row.q_start)
            step = 1 if row.q_end >= row.q_start else -1
            columns = bases
            for token in re.findall(r"\d+|..", str(row.btop)):
                if token.isdigit():
                    size = int(token)
                    if size:
                        left, right = sorted([position, position + step * (size - 1)])
                        matches += sum(max(0, min(right, b) - max(left, a) + 1) for a, b in remaining)
                    position += step * size
                elif token[0] == "-":
                    # A gap in the query consumes an alignment column but no query base.
                    anchor = min(end, max(start, position))
                    columns += any(a <= anchor <= b for a, b in remaining)
                else:
                    position += step
            alignment_columns += columns
        else:
            # Read historical 11-column reports, explicitly marking estimated identity.
            traceback = False
            columns = bases * float(row.align_len) / (end - start + 1)
            matches += columns * float(row.pident) / 100
            alignment_columns += columns
        covered = _merge_intervals(covered + remaining)
    query_bases = sum(end - start + 1 for start, end in covered)
    subject_intervals = [(int(row.s_start), int(row.s_end)) for row in alignments.itertuples()
                         if pd.notna(row.s_start) and pd.notna(row.s_end)]
    subject_bases = sum(end - start + 1 for start, end in _merge_intervals(subject_intervals)) if subject_intervals else np.nan
    first = alignments.iloc[0]
    return dict(aligned_bases=alignment_columns, aligned_query_bases=query_bases,
                aligned_subject_bases=subject_bases, n_hsps=len(alignments),
                pident=100 * matches / alignment_columns if alignment_columns else 0,
                query_coverage_percent=100 * query_bases / first.q_len,
                subject_coverage_percent=100 * subject_bases / first.s_len,
                best_evalue=alignments.evalue.min(),
                identity_evidence_scope="non-overlapping alignment columns" if traceback else "legacy HSP identity estimate")


def _read_blast_alignments(path):
    frame = _read_table(path, names=BLAST_FIELDS, header=None, dtype={"btop": "string"})
    return _to_numeric(frame, ["q_start", "q_end", "q_len", "s_len", "qcovs", "evalue",
                               "align_len", "pident", "s_start", "s_end", "bitscore"])


def _group_blast_alignments(frame, keys):
    rows = []
    for values, alignments in frame.groupby(keys, dropna=False, sort=False):
        rows.append(dict(zip(keys, values)) | _blast_alignment_summary(alignments))
    return pd.DataFrame(rows)


def _scope_host_hits(hits, provenance, assignments, own_host=True):
    if hits.empty:
        return hits.copy()
    owners = provenance.rename(columns={"original_id": "contig_id"})[["contig_id", "sample"]]
    scoped = hits.merge(owners.rename(columns={"contig_id": "contig"}), on="contig", how="inner")
    scoped["expected_host"] = scoped["sample"].map(assignments)
    matches = scoped["host"].eq(scoped["expected_host"])
    return scoped[scoped["expected_host"].notna() & (matches if own_host else ~matches)].copy()


def load_host_blast(paths, min_identity=90, min_query_coverage=90) -> pd.DataFrame:
    frames = []
    for item in _as_list(paths):
        path = Path(item)
        frame = _read_blast_alignments(path)
        if frame.empty:
            continue
        host_match = re.search(r"blastn_out_assembly_(.+?)\.tot\.csv$", path.name)
        frame["host"] = host_match.group(1).removesuffix("_viral_regions") if host_match else path.stem
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["contig", "host", "host_blast_like"])
    frame = pd.concat(frames, ignore_index=True)
    out = _group_blast_alignments(frame, ["query", "host", "subject", "subject_title", "q_len", "s_len"])
    out = out.rename(columns={"query": "contig", "subject": "host_contig", "subject_title": "host_title"})
    out["host_blast_like"] = out["pident"].ge(min_identity) & out["query_coverage_percent"].ge(min_query_coverage)
    return out.sort_values(["contig", "host_blast_like", "query_coverage_percent", "pident", "host", "host_contig"],
                           ascending=[True, False, False, False, True, True])


In [ ]:
_setup_plots()


def show_frame(title, df, n=20):
    display(Markdown(f"### {title}"))
    display(df.head(n) if hasattr(df, "head") else df)


In [ ]:
phase=snakemake.params.phase
sampling=str(snakemake.params.sampling)
samples=list(snakemake.params.samples)
results_dir=str(snakemake.params.results_dir)
if phase == "selection":
    sample_host_map={sample: snakemake.params.expected_host for sample in samples
                     if snakemake.params.expected_host != "not reported"}
else:
    sample_host_map=_load_sample_host_map(results_dir, snakemake.input.host_assignments)
display(pd.DataFrame([{"parameter": name, "value": value} for name, value in snakemake.params.items()]))


## Contig Catalogue And Retention

Exact MMseqs dereplication is followed by original CheckV clustering: independently >=95% ANI and >=85% aligned coverage of the shorter sequence. The hierarchy is original contig -> exact representative -> cluster -> cluster representative.

Retain a contig when depth >=5x AND (length >=4 kb OR depth >=10x), unless it matches its assigned host chromosome or host viral region at >=90% identity AND >=90% query coverage. Identity and query coverage describe the same non-overlapping alignment evidence. Other-host matches remain evidence only. Host assignments are mandatory when host FASTAs exist. The configured thresholds are shown above. geNomad, CheckV and terminal repeats are evidence, not eligibility gates. Host-prophage-associated sequences are excluded and flagged, not discarded from the catalogue.


In [ ]:
def add_contig_host_evidence(metadata, host_blast, sample_hosts=None):
    # Only the assigned host decides exclusion; other-host evidence remains reportable.
    if sample_hosts is not None:
        other = _scope_host_hits(host_blast, metadata, sample_hosts, own_host=False)
        other = other.sort_values(["host_blast_like", "query_coverage_percent", "pident"], ascending=False) if not other.empty else other
        other = other.reindex(columns=["contig", "host", "pident", "query_coverage_percent", "host_blast_like"])
        other = other.drop_duplicates("contig").rename(columns={
            "contig": "original_id", "host": "best_other_host_blast",
            "pident": "other_host_blast_identity_percent",
            "query_coverage_percent": "other_host_blast_query_coverage_percent", "host_blast_like": "other_host_match",
        })
        other["original_id"] = other["original_id"].astype(object)
        metadata = metadata.merge(other, on="original_id", how="left")
        metadata["other_host_match"] = metadata["other_host_match"].astype("boolean").fillna(False).astype(bool)
        host_blast = _scope_host_hits(host_blast, metadata, sample_hosts)
    # Use the existing >=90% identity AND >=90% query coverage host decisions.
    host = host_blast.sort_values(
        ["contig", "host_blast_like", "query_coverage_percent", "pident"],
        ascending=[True, False, False, False],
    ) if not host_blast.empty else host_blast
    host = host.reindex(columns=["contig", "host", "pident", "query_coverage_percent", "host_blast_like"])
    host = host.drop_duplicates("contig").rename(columns={
        "contig": "original_id", "host": "best_host_blast",
        "pident": "host_blast_identity_percent", "query_coverage_percent": "host_blast_query_coverage_percent",
        "host_blast_like": "host_match",
    })
    host["original_id"] = host["original_id"].astype(object)
    metadata = metadata.merge(host, on="original_id", how="left")
    metadata["host_match"] = metadata["host_match"].astype("boolean").fillna(False).astype(bool)
    return metadata


def build_global_contig_catalogue(provenance, exact_clusters, mosaic_clusters, genomad, checkv, contamination, host_blast, sample_hosts=None):
    # Original IDs are the current assembly FASTA IDs, before dereplication.
    metadata = provenance.rename(columns={"contig_id": "original_id"}).copy()
    metadata = metadata.merge(exact_clusters, on="original_id", how="left")
    mosaic_clusters = mosaic_clusters.copy()
    mosaic_clusters["exact_rep"] = mosaic_clusters["exact_rep"].str.split(",")
    mosaic_clusters = mosaic_clusters.explode("exact_rep").drop_duplicates()
    metadata = metadata.merge(mosaic_clusters, on="exact_rep", how="left")
    metadata["mosaic_cluster"] = metadata["cluster_rep"]
    metadata = metadata.merge(genomad.rename(columns={"contig_id": "original_id"}), on="original_id", how="left")

    metadata = add_contig_host_evidence(metadata, host_blast, sample_hosts)
    for field in ["host_match", "genomad_virus_call", "genomad_plasmid_call", "phage_plasmid_candidate"]:
        values = metadata.get(field, pd.Series(False, index=metadata.index))
        metadata[field] = values.astype("string").str.lower().isin(["true", "yes", "1"])

    metadata["phage_plasmid_evidence"] = [
        " + ".join(label for label, present in [
            ("geNomad viral call", row["genomad_virus_call"]),
            ("geNomad plasmid call", row["genomad_plasmid_call"]),
            ("geNomad virus hallmark genes", pd.to_numeric(row.get("genomad_virus_hallmark_count"), errors="coerce") > 0),
            ("geNomad plasmid hallmark genes", pd.to_numeric(row.get("genomad_plasmid_hallmark_count"), errors="coerce") > 0),
        ] if present) if row["phage_plasmid_candidate"] else "not reported"
        for _, row in metadata.iterrows()
    ]

    # Each original keeps its own CheckV result, not the cluster representative's result.
    if not checkv.empty:
        quality = checkv.drop(columns=["sample"], errors="ignore").drop_duplicates("contig").set_index("contig").add_prefix("CheckV_")
        metadata = metadata.join(quality, on="original_id")
    checkv_fields = ["provirus", "proviral_length", "gene_count", "viral_genes", "host_genes",
                     "checkv_quality", "miuvig_quality", "completeness", "completeness_method", "contamination", "warnings"]
    for field in checkv_fields:
        if "CheckV_" + field not in metadata:
            metadata["CheckV_" + field] = np.nan
    if not contamination.empty:
        regions = contamination.drop_duplicates("contig_id").set_index("contig_id")
        regions = regions.drop(columns=[name for name in regions if "CheckV_" + name in metadata])
        metadata = metadata.join(regions.add_prefix("CheckV_"), on="original_id")

    cluster_fields = ["mosaic_cluster", "cluster_rep", "number_exact_representatives", "number_original_contigs",
                      "number_samples"] + [
                      "number_genomad_virus", "fraction_genomad_virus", "number_genomad_plasmid",
                      "fraction_genomad_plasmid", "number_host_matching", "number_phage_plasmid_candidate",
                      "best_CheckV_quality", "maximum_CheckV_completeness"]
    quality_order = {"Complete": 0, "High-quality": 1, "Medium-quality": 2,
                     "Low-quality": 3, "Not-determined": 4}
    summaries = []
    for cluster, members in metadata.groupby("mosaic_cluster", sort=False):
        qualities = members["CheckV_checkv_quality"].dropna()
        known = qualities[qualities.isin(quality_order)]
        result = {
            "mosaic_cluster": cluster, "cluster_rep": cluster,
            "number_exact_representatives": members["exact_rep"].nunique(),
            "number_original_contigs": len(members), "number_samples": members["sample"].nunique(),
            "number_genomad_virus": int(members["genomad_virus_call"].sum()),
            "fraction_genomad_virus": members["genomad_virus_call"].mean(),
            "number_genomad_plasmid": int(members["genomad_plasmid_call"].sum()),
            "fraction_genomad_plasmid": members["genomad_plasmid_call"].mean(),
            "number_host_matching": int(members["host_match"].sum()),
            "number_phage_plasmid_candidate": int(members["phage_plasmid_candidate"].sum()),
            "best_CheckV_quality": min(known, key=quality_order.get) if not known.empty else "not reported",
            "maximum_CheckV_completeness": pd.to_numeric(members["CheckV_completeness"], errors="coerce").max(),
        }
        summaries.append(result)
    cluster_metadata = pd.DataFrame(summaries, columns=cluster_fields)
    first = ["sample", "original_id", "length_bp", "assembler", "exact_rep", "mosaic_cluster", "cluster_rep",
             "host_match", "phage_plasmid_candidate"]
    metadata = metadata.reindex(columns=first + [name for name in metadata if name not in first])
    return metadata.fillna("not reported"), cluster_metadata.fillna("not reported")


def add_catalogue_sourmash(metadata, clusters, queries, gather, taxonomy, database):
    metadata = metadata.astype({"cluster_rep": object})
    clusters = clusters.astype({"cluster_rep": object})
    evidence = queries.astype({"cluster_rep": object})
    taxonomy = taxonomy.reindex(columns=[
        "query_name", "status", "rank", "lineage", "fraction", "bp_match_at_rank", "query_ani_at_rank",
    ]).drop_duplicates("query_name").rename(columns={
        "query_name": "cluster_rep", "status": "sourmash_taxonomy_status",
        "rank": "sourmash_rank", "lineage": "sourmash_lineage",
        "fraction": "sourmash_taxonomy_fraction", "bp_match_at_rank": "sourmash_taxonomy_matched_bp",
        "query_ani_at_rank": "sourmash_taxonomy_estimated_ani",
    })
    taxonomy["cluster_rep"] = taxonomy["cluster_rep"].astype(object)
    evidence = evidence.merge(taxonomy, on="cluster_rep", how="left")
    match_fields = ["sourmash_best_match", "sourmash_best_match_accession", "sourmash_best_match_fraction",
                    "sourmash_total_match_fraction", "sourmash_estimated_shared_bp",
                    "sourmash_shared_hashes", "sourmash_number_reference_matches"]
    if not gather.empty:
        gather = gather.rename(columns={"match_name": "name"}).copy()
        for field in ["f_unique_to_query", "unique_intersect_bp", "scaled"]:
            gather[field] = pd.to_numeric(gather.get(field, pd.Series(index=gather.index, dtype=float)), errors="coerce")
        gather["shared_hashes"] = gather["unique_intersect_bp"] / gather["scaled"]
        best = gather.sort_values(["query_name", "f_unique_to_query"], ascending=[True, False])
        best = best.drop_duplicates("query_name").set_index("query_name")
        matches = gather.groupby("query_name").agg(
            sourmash_total_match_fraction=("f_unique_to_query", "sum"),
            sourmash_estimated_shared_bp=("unique_intersect_bp", "sum"),
            sourmash_shared_hashes=("shared_hashes", "sum"),
            sourmash_number_reference_matches=("name", "size"),
        )
        matches["sourmash_best_match"] = best["name"]
        matches["sourmash_best_match_accession"] = best["name"].astype(str).str.split().str[0]
        matches["sourmash_best_match_fraction"] = best["f_unique_to_query"]
        evidence = evidence.merge(matches.rename_axis("cluster_rep").reset_index(), on="cluster_rep", how="left")
    for field in match_fields:
        if field not in evidence:
            evidence[field] = np.nan
    no_match = evidence["sourmash_screen_status"].eq("screened") & evidence["sourmash_number_reference_matches"].isna()
    evidence.loc[no_match, "sourmash_screen_status"] = "no_reported_match"
    evidence["sourmash_gtdb_match"] = evidence["sourmash_taxonomy_status"].eq("match")
    evidence["sourmash_database"] = database
    evidence["sourmash_evidence_scope"] = "cluster_representative"
    evidence = evidence.fillna("not reported")
    return (metadata.merge(evidence, on="cluster_rep", how="left"),
            clusters.merge(evidence, on="cluster_rep", how="left"))


all_contig_metadata = pd.DataFrame()
global_cluster_metadata = pd.DataFrame()


In [ ]:
def select_isolate_contigs(metadata, coverage, host_viral, min_depth, min_length, short_depth, hosts):
    coverage=coverage.rename(columns={"contig": "original_id"}).drop(columns=["length", "host", "reference_set"], errors="ignore")
    metadata=metadata.merge(coverage, on=["original_id", "sample"], how="left")
    for column in ["mean_coverage", "covered_bases", "read_count", "rpkm", "breadth_percent"]:
        metadata[column]=pd.to_numeric(metadata.get(column, np.nan), errors="coerce").fillna(0)
    metadata["expected_host"]=metadata["sample"].map(sample_host_map).fillna("not reported")
    metadata["host_assessment"]=np.where(metadata["expected_host"].isin(hosts), "assessed", "not_assessed")
    metadata["host_match_scope"]=np.where(metadata["expected_host"].isin(hosts), "expected_host", "not_assessed")
    other_viral=_scope_host_hits(host_viral, metadata, sample_host_map, own_host=False)
    other_positive=set(other_viral.loc[other_viral["host_blast_like"], "contig"]) if not other_viral.empty else set()
    metadata["other_host_prophage_match"]=metadata["original_id"].isin(other_positive)
    host_viral=_scope_host_hits(host_viral, metadata, sample_host_map)
    host_positive=set(host_viral.loc[host_viral["host_blast_like"], "contig"]) if not host_viral.empty else set()
    metadata["host_prophage_match"]=metadata["original_id"].isin(host_positive)
    viral_best=host_viral.sort_values(["host_blast_like", "query_coverage_percent", "pident"], ascending=False) if not host_viral.empty else host_viral
    viral_best=viral_best.reindex(columns=["contig", "host", "host_contig", "pident", "query_coverage_percent"]).drop_duplicates("contig")
    viral_best=viral_best.rename(columns={"contig": "original_id", "host": "best_host_viral_reference",
        "host_contig": "host_viral_region", "pident": "host_viral_identity_percent",
        "query_coverage_percent": "host_viral_query_coverage_percent"})
    viral_best["original_id"]=viral_best["original_id"].astype(object)
    metadata=metadata.merge(viral_best, on="original_id", how="left")
    metadata["coverage_supported"]=metadata["mean_coverage"].ge(min_depth) & (
        metadata["length_bp"].ge(min_length) | metadata["mean_coverage"].ge(short_depth))
    metadata["retained"]=metadata["coverage_supported"] & ~metadata["host_match"] & ~metadata["host_prophage_match"]
    metadata["contig_set"]=np.where(metadata["retained"], "retained", "excluded")
    metadata["exclusion_reason"]=[
        "; ".join(reason for reason, applies in [
            ("below_minimum_depth", row.mean_coverage < min_depth),
            ("short_contig_below_required_depth", row.length_bp < min_length and row.mean_coverage < short_depth),
            ("host_prophage_associated", row.host_prophage_match),
            ("host_chromosome_match", row.host_match and not row.host_prophage_match),
        ] if applies) or "not excluded"
        for row in metadata.itertuples()
    ]
    return metadata


if phase == "selection":
    from Bio import SeqIO

    with open(snakemake.input.fasta) as handle:
        metadata=pd.DataFrame([
            {"sample": samples[0], "original_id": record.id, "length_bp": len(record)}
            for record in SeqIO.parse(handle, "fasta")
        ], columns=["sample", "original_id", "length_bp"])
    host_blast=load_host_blast(snakemake.input.host_blast, snakemake.params.host_min_identity, snakemake.params.host_min_query_coverage)
    host_viral=load_host_blast(snakemake.input.host_viral_blast, snakemake.params.host_min_identity, snakemake.params.host_min_query_coverage)
    metadata=add_contig_host_evidence(metadata, host_blast, sample_host_map)
    coverage=load_covstats([snakemake.input.assembled_covstats], "assembled", sampling)
    metadata=select_isolate_contigs(metadata, coverage, host_viral, snakemake.params.min_depth,
                                   snakemake.params.min_length, snakemake.params.short_depth, snakemake.params.hosts)
    _ensure_parent(snakemake.output.selection)
    metadata.to_csv(snakemake.output.selection, sep="\t", index=False, na_rep="not reported")
    show_frame("Sample contig selection: coverage and assigned-host evidence", metadata, n=30)


if phase == "catalogue":
    provenance=_read_table(snakemake.input.catalogue_provenance)
    CONTIG_SAMPLES=provenance.set_index("contig_id")["sample"].to_dict()
    exact=_read_table(snakemake.input.catalogue_exact_clusters, names=["exact_rep", "original_id"], header=None)
    mosaic=_read_table(snakemake.input.catalogue_mosaic_clusters, names=["cluster_rep", "exact_rep"], header=None)
    genomad=_read_table(snakemake.input.catalogue_genomad)
    checkv=_concat_nonempty([load_checkv(path) for path in snakemake.input.catalogue_checkv])
    contamination=_concat_nonempty([_read_table(path) for path in snakemake.input.catalogue_contamination])
    host_blast=load_host_blast(snakemake.input.host_blast, snakemake.params.host_min_identity, snakemake.params.host_min_query_coverage)
    host_viral=load_host_blast(snakemake.input.host_viral_blast, snakemake.params.host_min_identity, snakemake.params.host_min_query_coverage)
    all_contig_metadata, cluster_metadata=build_global_contig_catalogue(
        provenance, exact, mosaic, genomad, checkv, contamination, host_blast, sample_host_map)
    # Selection depends only on each assembly, its coverage and its assigned host.
    # Global annotations and other-host evidence are reporting inputs, not mapping inputs.
    selection_frames=[_read_table(path) for path in snakemake.input.selection]
    selection=pd.concat(selection_frames, ignore_index=True) if selection_frames else pd.DataFrame(columns=["sample", "original_id", "retained", "host_match"])
    other_fields=["best_other_host_blast", "other_host_blast_identity_percent",
                  "other_host_blast_query_coverage_percent", "other_host_match", "other_host_prophage_match"]
    selection=selection.drop(columns=other_fields + ["length_bp"], errors="ignore")
    all_contig_metadata=all_contig_metadata.drop(
        columns=[name for name in selection if name not in ["sample", "original_id"]], errors="ignore")
    all_contig_metadata=all_contig_metadata.merge(selection, on=["sample", "original_id"], how="left")
    other_viral=_scope_host_hits(host_viral, all_contig_metadata, sample_host_map, own_host=False)
    other_positive=set(other_viral.loc[other_viral["host_blast_like"], "contig"]) if not other_viral.empty else set()
    all_contig_metadata["other_host_prophage_match"]=all_contig_metadata["original_id"].isin(other_positive)
    if snakemake.input.catalogue_sourmash_queries:
        all_contig_metadata, cluster_metadata=add_catalogue_sourmash(
            all_contig_metadata, cluster_metadata, _read_table(snakemake.input.catalogue_sourmash_queries[0]),
            _read_table(snakemake.input.catalogue_sourmash_gather[0], sep=","),
            _read_table(snakemake.input.catalogue_sourmash_tax[0], sep=","),
            snakemake.params.sourmash_contig_catalogue_database)
    counts=all_contig_metadata.groupby("cluster_rep").agg(number_retained=("retained", "sum"),
        number_original_contigs=("retained", "size"), number_host_matching=("host_match", "sum"))
    counts["number_excluded"]=counts["number_original_contigs"] - counts["number_retained"]
    cluster_metadata=cluster_metadata.drop(columns="number_host_matching").merge(
        counts.drop(columns="number_original_contigs"), on="cluster_rep", how="left")
    for table, path in [(all_contig_metadata, snakemake.output.all_contig_metadata),
                        (cluster_metadata, snakemake.output.cluster_metadata)]:
        _ensure_parent(path)
        table.to_csv(path, sep="\t", index=False, na_rep="not reported")
    pd.DataFrame({"sample": samples, "expected_host": [sample_host_map.get(sample, "not reported") for sample in samples]}).to_csv(
        snakemake.output.sample_hosts, sep="\t", index=False)
    show_frame("All saved contigs: evidence and selection", all_contig_metadata, n=30)
    show_frame("All-contig clusters (not all are viral)", cluster_metadata, n=30)
